# ND2 Colocalization Analysis — Full Pipeline

**What this notebook does:**
- Loads ND2 files (single or batch folder, including subfolders)
- Analyses colocalization **slice-by-slice through the Z-stack** (no information lost) OR via Max Projection
- Rotates one channel at 90°/180°/270° to establish random colocalization baselines
- Computes **Pearson's R**, **Manders' C1 (M1)**, **Manders' C2 (M2)**, **(C1+C2)/2**, and **signal volume (µm³)**
- Saves rotated images as **TIFF** files for manual inspection
- Organises results into **N1/N2/N3 replicate folders**
- Generates per-file and **per-cell-line summary Excel files** with averages across replicates
- Produces publication-style figures matching your % overlap + volume layout

---
### First-time setup — run once in Terminal:
```
pip install nd2 numpy scipy scikit-image matplotlib pandas openpyxl tifffile
```

## Cell 1 — Import libraries
Run once, never edit.

In [ ]:
import nd2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from scipy.ndimage import rotate, gaussian_filter
from scipy.stats import pearsonr
from skimage.filters import threshold_otsu, threshold_triangle
from skimage.morphology import remove_small_objects, binary_opening, disk
from skimage.exposure import rescale_intensity
import tifffile
import openpyxl
from IPython.display import display, HTML
import os, warnings
from pathlib import Path
warnings.filterwarnings('ignore')
print('All libraries loaded!')

## Cell 2 — ⭐ Settings (only cell you need to edit)
Edit `INPUT_PATH`, channel indices, replicate assignments, and cell line labels.

In [ ]:
# ============================================================
#  FILE / FOLDER
# ============================================================
# Paste path to a single .nd2 file OR a master folder
# All .nd2 files in subfolders will be found automatically
INPUT_PATH = "path name"   # <-- EDIT THIS

# Where to save results (leave '' to save next to your ND2 files)
OUTPUT_DIR = ''

# ============================================================
#  CELL LINE DETECTION — from filename
# ============================================================
# The script will scan each filename for these keywords and
# assign the matching cell line label.
# Keys   = string to search for in the filename (case-insensitive)
# Values = label to use in results and folder names
#
# Based on your files:
#   BLEB_000_08172023_CTXDIFFD35_Control_SYP... → "Control"
#   BLEB_000_08172023_CTXDIFFD35_G32A_SYP...    → "G32A"
#   BLEB_000_08172023_CTXDIFFD35_R403C_SYP...   → "R403C"
CELL_LINE_KEYWORDS = {
    'Control' : 'Control',   # matches "Control" in filename
    'Ctrl'    : 'Control',   # matches "Ctrl" in filename — same label
    'G32A'    : 'G32A',
    'R403C'   : 'R403C',
}

# ============================================================
#  REPLICATE ORGANISATION
# ============================================================
# Map each SUBFOLDER NAME to a replicate label (N1, N2, N3).
# The script checks the folder the file lives in.
# If a file is not in any listed folder it goes to 'N_unassigned'.
#
# Example — if your structure is:
#   Master/
#     Experiment_Aug2023/   → N1
#     Experiment_Sep2023/   → N2
#     Experiment_Oct2023/   → N3
REPLICATE_MAP = {
    'N1 BLEB': 'N1',   # <-- edit folder names to match yours
    'N2 BLEB': 'N2',
    'N3 BLEB': 'N3',
}
# TIP: If all your files are already in one folder per replicate,
# just put the exact folder name here. You can also add more entries.

# ============================================================
#  CHANNELS  (0-based: 0 = first channel, 1 = second, etc.)
# ============================================================
# From your filenames: SYN(488) = channel 0, PSD95(568) = channel 1
# Adjust if your channel order differs.
CHANNEL_FIXED  = 1    # PSD95  — kept fixed
CHANNEL_ROTATE = 0    # SYP    — rotated for randomisation

# Human-readable labels for figures and Excel columns
CH1_LABEL = 'PSD95'
CH2_LABEL = 'SYP'

# ============================================================
#  Z-STACK HANDLING
# ============================================================
# 'slicewise' = analyse every Z-slice then average (recommended)
# 'max'       = max-intensity projection before analysis
Z_MODE = 'slicewise'

# Voxel size — read automatically from ND2 metadata.
# Only set these if your file lacks metadata (leave as None for auto).
VOXEL_XY_UM_OVERRIDE = None   # e.g. 0.1625  (None = read from file)
VOXEL_Z_UM_OVERRIDE  = None   # e.g. 0.5     (None = read from file)

# ============================================================
#  ROTATION ANGLES FOR RANDOMISATION
# ============================================================
ROTATION_ANGLES = [90, 180, 270]

# ============================================================
#  THRESHOLDING
# ============================================================
# 'triangle' = Triangle algorithm — good for sparse puncta with background
# 'otsu'     = Otsu's method
# 'manual'   = fixed value (set MANUAL_THRESHOLD below)
THRESHOLD_METHOD = 'triangle'
MANUAL_THRESHOLD = 50

# Minimum object size in pixels — removes specks after thresholding
MIN_OBJECT_SIZE = 10

# Morphological opening radius — further cleans noise (0 = off)
OPENING_RADIUS = 1

# Gaussian smoothing before thresholding (0 = off)
GAUSSIAN_SIGMA = 0.8

# ============================================================
print('Settings saved!')
print(f'  Input          : {INPUT_PATH}')
print(f'  Z mode         : {Z_MODE}')
print(f'  Threshold      : {THRESHOLD_METHOD}')
print(f'  Fixed channel  : [{CHANNEL_FIXED}] {CH1_LABEL}')
print(f'  Rotated channel: [{CHANNEL_ROTATE}] {CH2_LABEL}')
print(f'  Cell lines     : {list(CELL_LINE_KEYWORDS.values())}')
print(f'  Voxel overrides: XY={VOXEL_XY_UM_OVERRIDE}  Z={VOXEL_Z_UM_OVERRIDE}  (None = auto from metadata)')


## Cell 3 — Helper functions
Run once, no edits needed.

In [ ]:
# ── ND2 loading ───────────────────────────────────────────────
def load_nd2(filepath):
    with nd2.ND2File(filepath) as f:
        data = f.asarray()
        # ── Extract voxel sizes from metadata ──
        vx_xy, vx_z = None, None
        try:
            vx = f.voxel_size()   # returns (z, y, x) in µm
            vx_xy = float(vx.x)   # x and y are always equal
            vx_z  = float(vx.z)
        except Exception:
            pass
        # Fallback: try calibration from metadata directly
        if vx_xy is None or vx_xy == 0:
            try:
                cal = f.metadata.channels[0].volume.axesCalibration
                vx_xy = float(cal[0])   # X
                vx_z  = float(cal[2]) if len(cal) > 2 else None
            except Exception:
                pass
        meta = {
            'shape'   : data.shape,
            'axes'    : f.sizes,
            'channels': f.metadata.channels if hasattr(f.metadata, 'channels') else [],
            'vx_xy'   : vx_xy,
            'vx_z'    : vx_z,
        }
    return data, meta

def get_channel_names(meta, n_channels):
    if meta['channels']:
        return [ch.channel.name for ch in meta['channels']]
    return [f'Channel {i}' for i in range(n_channels)]

# ── Axis normalisation → always returns (C, Z, Y, X) ─────────
def to_czyx(data, axes):
    keys = list(axes.keys())
    target = [k for k in ['C','Z','Y','X'] if k in keys]
    current_order = [k for k in keys if k in ['C','Z','Y','X']]
    if current_order != target:
        perm = [current_order.index(t) for t in target]
        data = np.transpose(data, perm)
    if 'Z' not in keys:
        data = np.expand_dims(data, axis=1)
    return data  # (C, Z, Y, X)

# ── Thresholding with noise cleanup ──────────────────────────
def smart_threshold(img_2d, method, manual, min_obj, opening_r, sigma):
    img = img_2d.astype(float)
    if sigma > 0:
        img = gaussian_filter(img, sigma=sigma)
    if method == 'triangle':
        t = threshold_triangle(img)
    elif method == 'otsu':
        t = threshold_otsu(img)
    elif method == 'manual':
        img = rescale_intensity(img, out_range=(0, 255))
        t = manual
    else:
        return np.ones_like(img_2d, dtype=bool)
    mask = img > t
    if min_obj > 0 and mask.sum() > 0:
        mask = remove_small_objects(mask, min_size=min_obj)
    if opening_r > 0:
        mask = binary_opening(mask, footprint=disk(opening_r))
    return mask

# ── Colocalization per 2D slice ───────────────────────────────
def coloc_2d(ch1_slice, ch2_slice, mask1, mask2):
    joint = mask1 & mask2
    p1 = ch1_slice[joint].ravel().astype(float)
    p2 = ch2_slice[joint].ravel().astype(float)
    r = pearsonr(p1, p2)[0] if len(p1) >= 10 else np.nan
    denom1 = float(ch1_slice[mask1].sum())
    denom2 = float(ch2_slice[mask2].sum())
    m1 = float(ch1_slice[joint].sum()) / denom1 if denom1 > 0 else np.nan
    m2 = float(ch2_slice[joint].sum()) / denom2 if denom2 > 0 else np.nan
    return r, m1, m2

# ── Volume calculation ─────────────────────────────────────────
def calc_volume_um3(mask_zyx, vx_xy, vx_z):
    return float(mask_zyx.sum()) * vx_xy * vx_xy * vx_z

# ── Rotate a full Z-stack (Z,Y,X) ────────────────────────────
def rotate_zstack(zyx, angle):
    return np.stack([
        rotate(zyx[z], angle=angle, reshape=False, order=3, mode='reflect')
        for z in range(zyx.shape[0])
    ])

# ── Find ND2 files recursively ────────────────────────────────
def find_nd2_files(root):
    result = []
    for dp, dirs, files in os.walk(root):
        for f in files:
            if f.lower().endswith('.nd2'):
                result.append(os.path.join(dp, f))
    return sorted(result)

# ── Cell line detection FROM FILENAME ────────────────────────
def get_cell_line(filepath, keyword_map, default='unknown'):
    """
    Scan the filename for cell line keywords.
    keyword_map = {'Control': 'Control', 'G32A': 'G32A', 'R403C': 'R403C'}
    Matching is case-insensitive. Returns the label for the first match found.
    If multiple keywords match, the one that appears earliest in the filename wins.
    """
    fname = Path(filepath).name
    matches = []
    for keyword, label in keyword_map.items():
        idx = fname.lower().find(keyword.lower())
        if idx != -1:
            matches.append((idx, label))
    if matches:
        matches.sort(key=lambda x: x[0])
        return matches[0][1]
    return default

# ── Replicate detection FROM FOLDER NAME ─────────────────────
def get_replicate(filepath, replicate_map, default='N_unassigned'):
    """
    Check the parent folder(s) of the file against replicate_map keys.
    Searches from immediate parent upward so nested folders work too.
    """
    parts = Path(filepath).parts
    for part in reversed(parts[:-1]):   # skip filename itself
        if part in replicate_map:
            return replicate_map[part]
    return default

# ── Preview what will be detected ────────────────────────────
def preview_detection(nd2_files, cell_line_keywords, replicate_map,
                      xy_override=None, z_override=None):
    """Print a table showing how each file will be labelled + voxel sizes."""
    print(f"{'File':<58} {'Cell Line':<12} {'Replicate':<15} {'XY (µm)':<12} {'Z (µm)'}")
    print('-' * 110)
    for fp in nd2_files:
        cl  = get_cell_line(fp, cell_line_keywords)
        rep = get_replicate(fp, replicate_map)
        # Read voxel sizes from this file
        try:
            _, m = load_nd2(fp)
            vxy = xy_override if xy_override else m['vx_xy']
            vz  = z_override  if z_override  else m['vx_z']
            vxy_str = f'{vxy:.4f}' if vxy else 'NOT FOUND'
            vz_str  = f'{vz:.4f}'  if vz  else 'NOT FOUND'
        except Exception as e:
            vxy_str = vz_str = f'ERROR: {e}'
        print(f"{Path(fp).name:<58} {cl:<12} {rep:<15} {vxy_str:<12} {vz_str}")

print('Helper functions loaded!')
print()
print('TIP: Run the preview in Cell 5a to confirm cell line detection before running the full analysis.')


## Cell 5a — Preview file detection (optional but recommended)
Run this **before** Cell 6 to confirm your filenames are being parsed correctly.
No files are processed — it just prints a table.

In [ ]:
# Finds all ND2s and shows how each will be labelled + voxel sizes from metadata
input_path = Path(INPUT_PATH)
if not input_path.exists():
    print('ERROR: INPUT_PATH not found — edit Cell 2 first.')
else:
    _files = find_nd2_files(str(input_path)) if input_path.is_dir() else [str(input_path)]
    print(f'Found {len(_files)} ND2 file(s)\n')
    preview_detection(_files, CELL_LINE_KEYWORDS, REPLICATE_MAP,
                      xy_override=VOXEL_XY_UM_OVERRIDE,
                      z_override=VOXEL_Z_UM_OVERRIDE)
    print('\nIf XY or Z shows NOT FOUND, set VOXEL_XY_UM_OVERRIDE / VOXEL_Z_UM_OVERRIDE in Cell 2.')
    print('If any cell line shows "unknown" or replicate shows "N_unassigned", update Cell 2.')


## Cell 4 — Core analysis + saving functions
Run once, no edits needed.

In [ ]:
def analyse_file(filepath, out_dir,
                 ch1_idx, ch2_idx, ch1_label, ch2_label,
                 rotation_angles, z_mode,
                 thresh_method, manual_thresh,
                 min_obj, opening_r, sigma,
                 vx_xy, vx_z,
                 save_tiffs=True):
    """
    Full analysis for one ND2 file.
    Returns list of result dicts (one per rotation condition).
    """
    print(f'  Loading...')
    data, meta = load_nd2(filepath)

    # Reorder to (C, Z, Y, X)
    data = to_czyx(data, meta['axes'])
    n_channels = data.shape[0]
    n_slices   = data.shape[1]
    print(f'  Shape (C,Z,Y,X): {data.shape}')

    channel_names = get_channel_names(meta, n_channels)
    print(f'  Channels: {channel_names}')

    if ch1_idx >= n_channels or ch2_idx >= n_channels:
        print(f'  SKIP: channel index out of range')
        return None

    # Extract (Z, Y, X) stacks for each channel
    zyx1 = data[ch1_idx].astype(float)
    zyx2 = data[ch2_idx].astype(float)

    stem = Path(filepath).stem

    def process_pair(stack1, stack2, label):
        """
        Given two (Z,Y,X) stacks, compute all metrics.
        If z_mode='slicewise', average across slices.
        If z_mode='max'/'mean', project first.
        Returns dict of metrics + saves TIFF.
        """
        if z_mode == 'slicewise':
            rs, m1s, m2s = [], [], []
            mask1_3d = np.zeros_like(stack1, dtype=bool)
            mask2_3d = np.zeros_like(stack2, dtype=bool)

            for z in range(n_slices):
                sl1 = stack1[z]; sl2 = stack2[z]
                mk1 = smart_threshold(sl1, thresh_method, manual_thresh,
                                      min_obj, opening_r, sigma)
                mk2 = smart_threshold(sl2, thresh_method, manual_thresh,
                                      min_obj, opening_r, sigma)
                mask1_3d[z] = mk1; mask2_3d[z] = mk2
                r, m1, m2 = coloc_2d(sl1, sl2, mk1, mk2)
                rs.append(r); m1s.append(m1); m2s.append(m2)

            pearson = float(np.nanmean(rs))
            manders_c1 = float(np.nanmean(m1s))   # C1 over C2
            manders_c2 = float(np.nanmean(m2s))   # C2 over C1
            pct_overlap_c1 = manders_c1 * 100
            pct_overlap_c2 = manders_c2 * 100
            vol1 = calc_volume_um3(mask1_3d, vx_xy, vx_z)
            vol2 = calc_volume_um3(mask2_3d, vx_xy, vx_z)

        else:  # max or mean projection
            fn = np.max if z_mode == 'max' else np.mean
            proj1 = fn(stack1, axis=0)
            proj2 = fn(stack2, axis=0)
            mk1 = smart_threshold(proj1, thresh_method, manual_thresh,
                                  min_obj, opening_r, sigma)
            mk2 = smart_threshold(proj2, thresh_method, manual_thresh,
                                  min_obj, opening_r, sigma)
            pearson, manders_c1, manders_c2 = coloc_2d(proj1, proj2, mk1, mk2)
            pct_overlap_c1 = manders_c1 * 100
            pct_overlap_c2 = manders_c2 * 100
            # Volume from full 3D stack using same threshold
            mask1_3d = np.stack([
                smart_threshold(stack1[z], thresh_method, manual_thresh,
                                min_obj, opening_r, sigma)
                for z in range(n_slices)])
            mask2_3d = np.stack([
                smart_threshold(stack2[z], thresh_method, manual_thresh,
                                min_obj, opening_r, sigma)
                for z in range(n_slices)])
            vol1 = calc_volume_um3(mask1_3d, vx_xy, vx_z)
            vol2 = calc_volume_um3(mask2_3d, vx_xy, vx_z)

        avg_manders = (manders_c1 + manders_c2) / 2
        pct_avg     = avg_manders * 100

        # Save rotated channel as TIFF
        if save_tiffs and label != 'Baseline':
            tiff_path = os.path.join(out_dir, f'{stem}_{label.replace(" ","_")}_ch2_rotated.tif')
            tifffile.imwrite(tiff_path, stack2.astype(np.float32),
                             imagej=True,
                             resolution=(1/vx_xy, 1/vx_xy),
                             metadata={'spacing': vx_z, 'unit': 'um', 'axes': 'ZYX'})
            print(f'    TIFF saved: {Path(tiff_path).name}')

        return {
            'condition'                      : label,
            'pearson_r'                      : round(pearson, 4),
            f'manders_C1_{ch1_label}_over_{ch2_label}': round(manders_c1, 4),
            f'manders_C2_{ch2_label}_over_{ch1_label}': round(manders_c2, 4),
            f'manders_avg_(C1+C2)/2'         : round(avg_manders, 4),
            f'pct_overlap_C1_{ch1_label}_%'  : round(pct_overlap_c1, 2),
            f'pct_overlap_C2_{ch2_label}_%'  : round(pct_overlap_c2, 2),
            f'pct_overlap_avg_(C1+C2)/2_%'   : round(pct_avg, 2),
            f'vol_{ch1_label}_um3'           : round(vol1, 3),
            f'vol_{ch2_label}_um3'           : round(vol2, 3),
        }

    # Baseline
    print(f'  Baseline...')
    rows = [process_pair(zyx1, zyx2, 'Baseline')]

    # Rotations
    for angle in rotation_angles:
        print(f'  Rotation {angle}°...')
        zyx2_rot = rotate_zstack(zyx2, angle)
        rows.append(process_pair(zyx1, zyx2_rot, f'Rot_{angle}deg'))

    # Add file info to each row
    for row in rows:
        if row:
            row['file'] = Path(filepath).name

    return [r for r in rows if r is not None]


def save_excel(df, path, sheet_name='Results'):
    """Save DataFrame to Excel with auto-column widths."""
    with pd.ExcelWriter(path, engine='openpyxl') as writer:
        df.to_excel(writer, index=False, sheet_name=sheet_name)
        ws = writer.sheets[sheet_name]
        for col in ws.columns:
            max_len = max(len(str(cell.value or '')) for cell in col)
            ws.column_dimensions[col[0].column_letter].width = min(max_len + 2, 40)


def save_replicate_excel(rep_df, rep_path, rep_label):
    """
    Save one Excel file per replicate (e.g. N1_summary.xlsx) with:
      Sheet 1: All raw image data for this replicate
      Sheet 2+: One sheet per cell line showing every image row
      Final sheet: Mean ± SD per cell line × rotation condition
    """
    metric_cols = [c for c in rep_df.columns
                   if c not in ['file', 'condition', 'replicate', 'cell_line']]

    with pd.ExcelWriter(rep_path, engine='openpyxl') as writer:

        # Sheet 1 — all raw data for this replicate
        rep_df.to_excel(writer, index=False, sheet_name='All_Images')

        # One sheet per cell line — every individual image row
        for cl in sorted(rep_df['cell_line'].unique()):
            cl_df = rep_df[rep_df['cell_line'] == cl].copy()
            sheet_name = cl[:31]  # Excel sheet name limit
            cl_df.to_excel(writer, index=False, sheet_name=sheet_name)

        # Summary sheet — mean ± SD per cell_line × condition, within this replicate only
        grp = (rep_df.groupby(['cell_line', 'condition'])[metric_cols]
               .agg(['mean', 'std'])
               .round(4))
        # Flatten MultiIndex columns: (metric, mean) -> metric_mean
        grp.columns = ['_'.join(c) for c in grp.columns]
        grp = grp.reset_index()
        # Sort so Baseline comes first, then rotations
        cond_order = ['Baseline'] + [f'Rot_{a}deg' for a in sorted(
            [int(c.replace('Rot_','').replace('deg',''))
             for c in rep_df['condition'].unique() if c != 'Baseline'])]
        grp['_sort'] = grp['condition'].map(
            {c: i for i, c in enumerate(cond_order)})
        grp = grp.sort_values(['cell_line', '_sort']).drop(columns='_sort')
        grp.to_excel(writer, index=False, sheet_name=f'{rep_label}_CellLine_Means')

        # Auto-width all sheets
        for sheet in writer.sheets.values():
            for col in sheet.columns:
                max_len = max(len(str(cell.value or '')) for cell in col)
                sheet.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    print(f'  Replicate Excel saved: {rep_path}')


def make_summary_excel(all_df, out_path, ch1_label, ch2_label):
    """
    Master summary Excel across all replicates:
      Sheet 1 : All raw data (every image, every condition)
      Sheet 2 : Mean per replicate × cell_line × condition  (kept SEPARATE — not combined)
      Sheet 3 : Grand mean ± SD per cell_line × condition   (biological replicate summary)
    """
    metric_cols = [c for c in all_df.columns
                   if c not in ['file', 'condition', 'replicate', 'cell_line']]

    with pd.ExcelWriter(out_path, engine='openpyxl') as writer:

        # Sheet 1: all raw data
        all_df.to_excel(writer, index=False, sheet_name='All_Raw_Data')

        # Sheet 2: mean per replicate × cell_line × condition — each combination is its own row
        # This keeps N1/Control/Baseline separate from N2/Control/Baseline etc.
        grp2 = (all_df.groupby(['replicate', 'cell_line', 'condition'])[metric_cols]
                .mean().round(4).reset_index())
        # Sort: replicate → cell_line → Baseline first then rotations
        grp2.to_excel(writer, index=False, sheet_name='Means_Rep_CellLine_Cond')

        # Sheet 3: grand mean ± SD per cell_line × condition (across all replicates)
        grp3 = (all_df.groupby(['cell_line', 'condition'])[metric_cols]
                .agg(['mean', 'std']).round(4))
        grp3.columns = ['_'.join(c) for c in grp3.columns]
        grp3 = grp3.reset_index()
        grp3.to_excel(writer, index=False, sheet_name='Grand_Mean_SD_by_CellLine')

        # Auto-width
        for sheet in writer.sheets.values():
            for col in sheet.columns:
                max_len = max(len(str(cell.value or '')) for cell in col)
                sheet.column_dimensions[col[0].column_letter].width = min(max_len + 2, 45)

    print(f'  Master summary Excel saved: {out_path}')


print('Core functions loaded!')


## Cell 5 — Figure functions
Run once, no edits needed.

In [ ]:
def plot_overlap_dotplot(df, metric_col, ylabel, title, save_path=None):
    """
    Dot-plot matching your panel D style:
    X-axis = condition (Baseline, Rot_90deg, …)
    Colour = cell_line
    """
    fig, ax = plt.subplots(figsize=(9, 5))
    conditions  = df['condition'].unique()
    cell_lines  = df['cell_line'].unique()
    palette     = {'Control':'#4575b4','G32A':'#984ea3','R403C':'#4daf4a',
                   'unknown':'#888888'}

    x_positions = {c: i for i, c in enumerate(conditions)}
    jitter_w    = 0.12

    for cl in cell_lines:
        sub = df[df['cell_line'] == cl]
        color = palette.get(cl, '#888888')
        for cond in conditions:
            vals = sub[sub['condition']==cond][metric_col].dropna().values
            xc   = x_positions[cond]
            jitter = np.random.uniform(-jitter_w, jitter_w, len(vals))
            ax.scatter(xc + jitter, vals, color=color, s=40,
                       alpha=0.85, zorder=3, label=cl if cond==conditions[0] else '')
        # Mean ± SEM per condition
        for cond in conditions:
            vals = sub[sub['condition']==cond][metric_col].dropna().values
            if len(vals) == 0: continue
            xc  = x_positions[cond]
            mn  = np.mean(vals)
            sem = np.std(vals)/np.sqrt(len(vals))
            ax.errorbar(xc, mn, yerr=sem, fmt='_', color=color,
                        capsize=4, linewidth=2, markersize=14, zorder=4)

    ax.set_xticks(list(x_positions.values()))
    ax.set_xticklabels([c.replace('_',' ') for c in conditions], fontsize=10)
    ax.set_ylabel(ylabel, fontsize=11)
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    handles = [mpatches.Patch(color=palette.get(cl,'#888'), label=cl) for cl in cell_lines]
    ax.legend(handles=handles, frameon=False, fontsize=9)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show(); plt.close()


def plot_volume_bars(df, ch1_label, ch2_label, save_path=None):
    """
    Bar charts matching your panel E style:
    Vol Ch1 | Vol Ch2 | % Overlap avg
    grouped by cell_line, coloured by cell_line
    """
    cell_lines = df['cell_line'].unique()
    palette    = {'Control':'#4575b4','G32A':'#984ea3','R403C':'#4daf4a',
                  'unknown':'#888888'}
    vol1_col   = f'vol_{ch1_label}_um3'
    vol2_col   = f'vol_{ch2_label}_um3'
    pct_col    = 'pct_overlap_avg_(C1+C2)/2_%'

    # Baseline only for volume/overlap comparison
    base = df[df['condition']=='Baseline'].copy()

    fig, axes = plt.subplots(1, 3, figsize=(13, 5))
    titles = [f'{ch1_label} Volume', f'{ch2_label} Volume', f'{ch1_label}:{ch2_label} Overlap']
    cols   = [vol1_col, vol2_col, pct_col]
    ylabels= ['µm³', 'µm³', '% Overlap']

    x = np.arange(len(cell_lines))
    w = 0.55

    for ax, title, col, ylabel in zip(axes, titles, cols, ylabels):
        means, sems, colors = [], [], []
        for cl in cell_lines:
            vals = base[base['cell_line']==cl][col].dropna().values
            means.append(np.mean(vals) if len(vals)>0 else 0)
            sems.append(np.std(vals)/np.sqrt(len(vals)) if len(vals)>1 else 0)
            colors.append(palette.get(cl,'#888888'))

        bars = ax.bar(x, means, width=w, color=colors, alpha=0.8,
                      edgecolor='black', linewidth=0.7)
        ax.errorbar(x, means, yerr=sems, fmt='none', color='black',
                    capsize=4, linewidth=1.5)

        # Overlay individual dots
        for i, cl in enumerate(cell_lines):
            vals = base[base['cell_line']==cl][col].dropna().values
            jitter = np.random.uniform(-0.12, 0.12, len(vals))
            ax.scatter(i + jitter, vals, color=palette.get(cl,'#888'),
                       s=35, zorder=5, alpha=0.9, edgecolors='white', linewidths=0.4)

        ax.set_xticks(x)
        ax.set_xticklabels(cell_lines, fontsize=10)
        ax.set_ylabel(ylabel, fontsize=10)
        ax.set_title(title, fontsize=11, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=200, bbox_inches='tight')
    plt.show(); plt.close()


def plot_composite_tiff(zyx1, zyx2, zyx2_rot, angle,
                        ch1_label, ch2_label, save_path=None):
    """Max-project for visual comparison and show composite."""
    def norm(x):
        mn,mx=x.min(),x.max(); return (x-mn)/(mx-mn+1e-9)
    proj1   = zyx1.max(axis=0)
    proj2   = zyx2.max(axis=0)
    proj2r  = zyx2_rot.max(axis=0)

    fig = plt.figure(figsize=(16,4), facecolor='#111')
    gs  = gridspec.GridSpec(1,4,wspace=0.04)
    panels = [
        (proj1,  f'{ch1_label}\n(fixed)',        'RdPu'),
        (proj2,  f'{ch2_label}\n(original)',      'GnBu'),
        (proj2r, f'{ch2_label}\n(rotated {angle}°)','YlGn'),
    ]
    for i,(img,ttl,cmap) in enumerate(panels):
        ax=fig.add_subplot(gs[i])
        ax.imshow(norm(img),cmap=cmap); ax.set_title(ttl,color='w',fontsize=9,pad=4); ax.axis('off')
    ax4=fig.add_subplot(gs[3])
    ov=np.zeros((*proj1.shape,3),dtype=float)
    ov[:,:,0]=norm(proj1); ov[:,:,1]=norm(proj2r)
    ax4.imshow(ov); ax4.set_title(f'Overlay\n(rot {angle}°)',color='w',fontsize=9,pad=4); ax4.axis('off')
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight', facecolor='#111')
    plt.show(); plt.close()

print('Figure functions loaded!')

## Cell 6 — ▶ Run the analysis
Press Run. Everything is driven by the settings in Cell 2.

In [ ]:
np.random.seed(42)   # reproducible jitter

# ── Validate input path ──────────────────────────────────────
input_path = Path(INPUT_PATH)
if not input_path.exists():
    raise FileNotFoundError(
        f'Path not found:\n  {INPUT_PATH}\n\nEdit INPUT_PATH in Cell 2.')

# ── Find files ───────────────────────────────────────────────
if input_path.is_file() and input_path.suffix.lower() == '.nd2':
    nd2_files = [str(input_path)]
    print(f'Single file mode: {input_path.name}')
elif input_path.is_dir():
    nd2_files = find_nd2_files(str(input_path))
    print(f'Batch mode — {len(nd2_files)} ND2 files found:')
    for f in nd2_files: print(f'  {f}')
else:
    raise ValueError('INPUT_PATH must be an .nd2 file or folder.')

if not nd2_files:
    raise FileNotFoundError('No ND2 files found.')

# ── Output root ──────────────────────────────────────────────
base_out = Path(OUTPUT_DIR) if OUTPUT_DIR else (
    input_path.parent if input_path.is_file() else input_path)
base_out = base_out / 'colocalization_results'
base_out.mkdir(parents=True, exist_ok=True)
print(f'\nOutput root: {base_out}')

# ── Process each file ────────────────────────────────────────
all_rows = []

for i, filepath in enumerate(nd2_files):
    fname = Path(filepath).name
    rep   = get_replicate(filepath, REPLICATE_MAP)
    cl    = get_cell_line(filepath, CELL_LINE_KEYWORDS)

    print(f'\n=== [{i+1}/{len(nd2_files)}] {fname}  |  replicate={rep}  cell_line={cl} ===')

    # Per-file output inside replicate / cell_line subfolder
    file_out = base_out / rep / cl / Path(filepath).stem
    file_out.mkdir(parents=True, exist_ok=True)

    try:
        # ── Resolve voxel sizes: metadata first, override if set ──
        _tmp_data, _tmp_meta = load_nd2(filepath)
        vx_xy = VOXEL_XY_UM_OVERRIDE if VOXEL_XY_UM_OVERRIDE else _tmp_meta['vx_xy']
        vx_z  = VOXEL_Z_UM_OVERRIDE  if VOXEL_Z_UM_OVERRIDE  else _tmp_meta['vx_z']
        if not vx_xy or vx_xy == 0:
            print('  WARNING: XY voxel size not found in metadata. Set VOXEL_XY_UM_OVERRIDE in Cell 2.')
            vx_xy = 1.0
        if not vx_z or vx_z == 0:
            print('  WARNING: Z voxel size not found in metadata. Set VOXEL_Z_UM_OVERRIDE in Cell 2.')
            vx_z = 1.0
        print(f'  Voxel size: XY={vx_xy:.4f} µm  Z={vx_z:.4f} µm')
        del _tmp_data, _tmp_meta
        rows = analyse_file(
            filepath        = filepath,
            out_dir         = str(file_out),
            ch1_idx         = CHANNEL_FIXED,
            ch2_idx         = CHANNEL_ROTATE,
            ch1_label       = CH1_LABEL,
            ch2_label       = CH2_LABEL,
            rotation_angles = ROTATION_ANGLES,
            z_mode          = Z_MODE,
            thresh_method   = THRESHOLD_METHOD,
            manual_thresh   = MANUAL_THRESHOLD,
            min_obj         = MIN_OBJECT_SIZE,
            opening_r       = OPENING_RADIUS,
            sigma           = GAUSSIAN_SIGMA,
            vx_xy           = vx_xy,
            vx_z            = vx_z,
            save_tiffs      = True,
        )
        if rows:
            for row in rows:
                row['replicate'] = rep
                row['cell_line'] = cl
            df_file = pd.DataFrame(rows)

            # Per-file Excel
            xl_path = file_out / f'{Path(filepath).stem}_results.xlsx'
            save_excel(df_file, xl_path)
            print(f'  Excel: {xl_path}')

            # Composite figures
            data, meta = load_nd2(filepath)
            data = to_czyx(data, meta['axes'])
            zyx1 = data[CHANNEL_FIXED].astype(float)
            zyx2 = data[CHANNEL_ROTATE].astype(float)
            for angle in ROTATION_ANGLES:
                zyx2_rot = rotate_zstack(zyx2, angle)
                fig_path = file_out / f'{Path(filepath).stem}_composite_rot{angle}.png'
                plot_composite_tiff(zyx1, zyx2, zyx2_rot, angle,
                                    CH1_LABEL, CH2_LABEL, save_path=str(fig_path))

            all_rows.extend(rows)

    except Exception as e:
        import traceback
        print(f'  ERROR: {e}')
        traceback.print_exc()

# ── Master DataFrame ─────────────────────────────────────────
if not all_rows:
    print('No files processed successfully.')
else:
    master_df = pd.DataFrame(all_rows)

    # ── Per-replicate Excel: one file per replicate ───────────
    # Each file has: all raw images, one sheet per cell line,
    # and a summary sheet with mean ± SD per cell_line × condition
    # Cell lines are NEVER combined — each keeps its own rows/means
    for rep in sorted(master_df['replicate'].unique()):
        rep_df   = master_df[master_df['replicate'] == rep]
        rep_path = base_out / rep / f'{rep}_cellline_summary.xlsx'
        save_replicate_excel(rep_df, rep_path, rep)

    # ── Master summary Excel across all replicates ────────────
    summary_path = base_out / 'SUMMARY_all_replicates.xlsx'
    make_summary_excel(master_df, summary_path, CH1_LABEL, CH2_LABEL)

    print(f'\n=== DONE — {len(nd2_files)} files processed ===')
    print(f'All results: {base_out}')


## Cell 7 — Publication figures
Run after Cell 6. Generates panel D (dot-plot) and panel E (bar chart) style figures.

In [ ]:
if 'master_df' not in dir() or master_df is None:
    print('Run Cell 6 first.')
else:
    figs_out = base_out / 'figures'
    figs_out.mkdir(exist_ok=True)

    pct_c1_col  = f'pct_overlap_C1_{CH1_LABEL}_%'
    pct_c2_col  = f'pct_overlap_C2_{CH2_LABEL}_%'
    pct_avg_col = 'pct_overlap_avg_(C1+C2)/2_%'

    # Panel D style — % overlap across conditions (all rotations)
    plot_overlap_dotplot(
        master_df, pct_avg_col,
        ylabel=f'% Overlap ({CH1_LABEL}:{CH2_LABEL} avg)',
        title=f'{CH1_LABEL}:{CH2_LABEL} Overlap — Baseline vs Rotations',
        save_path=str(figs_out / 'panel_D_overlap_avg.png')
    )
    plot_overlap_dotplot(
        master_df, pct_c1_col,
        ylabel=f'% Overlap (C1 {CH1_LABEL} over {CH2_LABEL})',
        title=f'{CH1_LABEL} over {CH2_LABEL} — Baseline vs Rotations',
        save_path=str(figs_out / 'panel_D_overlap_C1.png')
    )
    plot_overlap_dotplot(
        master_df, pct_c2_col,
        ylabel=f'% Overlap (C2 {CH2_LABEL} over {CH1_LABEL})',
        title=f'{CH2_LABEL} over {CH1_LABEL} — Baseline vs Rotations',
        save_path=str(figs_out / 'panel_D_overlap_C2.png')
    )

    # Panel E style — volumes + overlap for baseline only, by cell line
    plot_volume_bars(
        master_df, CH1_LABEL, CH2_LABEL,
        save_path=str(figs_out / 'panel_E_volume_and_overlap.png')
    )

    print(f'Figures saved to: {figs_out}')

## Cell 8 — Results table preview
Run after Cell 6 to view a colour-coded table in the notebook.

In [ ]:
if 'master_df' not in dir():
    print('Run Cell 6 first.')
else:
    show_cols = ['file','replicate','cell_line','condition',
                 f'pct_overlap_C1_{CH1_LABEL}_%',
                 f'pct_overlap_C2_{CH2_LABEL}_%',
                 'pct_overlap_avg_(C1+C2)/2_%',
                 f'vol_{CH1_LABEL}_um3',
                 f'vol_{CH2_LABEL}_um3',
                 'pearson_r']
    show_cols = [c for c in show_cols if c in master_df.columns]
    fmt_cols  = {c: '{:.2f}' for c in show_cols if master_df[c].dtype == float}

    display(HTML('<h3>Results preview</h3>'))
    display(
        master_df[show_cols].style
        .format(fmt_cols)
        .background_gradient(subset=['pct_overlap_avg_(C1+C2)/2_%'],
                             cmap='YlOrRd', vmin=0, vmax=50)
        .background_gradient(subset=['pearson_r'], cmap='RdYlGn', vmin=-1, vmax=1)
    )